# ETL Pipeline - Retail Price Data
## Extract, Transform, Load Pipeline for Naivas & Quickmart Data

This notebook handles the complete ETL process:
1. **Extract**: Read data from Excel files
2. **Transform**: Clean and standardize data
3. **Load**: Insert into data warehouse

---

## 1. Import Libraries

In [7]:
import pandas as pd
import os
import re
from datetime import datetime

print("✓ Libraries imported successfully")
print(f"Pandas version: {pd.__version__}")

✓ Libraries imported successfully
Pandas version: 3.0.2


## 2. Define Helper Functions

In [8]:
def extract_date_from_filename(filename):
    """
    Extract date from filename handling multiple date formats.
    Supports:
    - YYYY-MM-DD (e.g., Naivas_CPI_2026-01-01.xlsx)
    - DD-MM-YYYY (e.g., Quickmart_02-02-2026.xlsx)
    """
    # Try YYYY-MM-DD format first
    match = re.search(r'(\d{4}-\d{2}-\d{2})', filename)
    if match:
        return pd.to_datetime(match.group(1), format='%Y-%m-%d')
    
    # Try DD-MM-YYYY format
    match = re.search(r'(\d{2}-\d{2}-\d{4})', filename)
    if match:
        return pd.to_datetime(match.group(1), format='%d-%m-%Y')
    
    # If no date found, return None
    print(f"Warning: Could not extract date from filename: {filename}")
    return None


def process_folder(folder_path, store_name):
    """
    ETL Function to Extract, Transform, and Load data from a folder.
    
    Extract: Read all Excel/CSV files from folder
    Transform: Add Date and Store columns, clean data
    Load: Concatenate into single DataFrame
    """
    all_data = []
    skipped_files = []
    
    print(f"\n{'='*50}")
    print(f"Processing {store_name} data from: {folder_path}")
    print(f"{'='*50}")
    
    for filename in os.listdir(folder_path):
        if filename.endswith(".xlsx") or filename.endswith(".csv"):
            try:
                # Extract date from filename
                file_date = extract_date_from_filename(filename)
                
                if file_date is None:
                    skipped_files.append(filename)
                    continue
                
                # Read the file
                file_path = os.path.join(folder_path, filename)
                if filename.endswith(".xlsx"):
                    df = pd.read_excel(file_path)
                else:
                    df = pd.read_csv(file_path)
                
                # Transform: Add metadata columns
                df['Date'] = file_date
                df['Store'] = store_name
                df['Source_File'] = filename
                
                all_data.append(df)
                print(f"✓ Processed: {filename} ({len(df)} rows)")
                
            except Exception as e:
                print(f"✗ Error processing {filename}: {str(e)}")
                skipped_files.append(filename)
    
    if skipped_files:
        print(f"\n⚠ Skipped {len(skipped_files)} files:")
        for f in skipped_files:
            print(f"  - {f}")
    
    if not all_data:
        raise ValueError(f"No valid data files found in {folder_path}")
    
    # Load: Combine all data
    result_df = pd.concat(all_data, ignore_index=True)
    print(f"\n✓ Total rows loaded: {len(result_df)}")
    print(f"✓ Date range: {result_df['Date'].min()} to {result_df['Date'].max()}")
    
    return result_df


print("✓ Helper functions defined")

✓ Helper functions defined


## 3. Extract - Load Raw Data from Files

In [9]:
print("Starting Data Extraction...")
print("="*70)

# Extract & Transform: Process both stores
df_naivas = process_folder('../Data/Naivas', 'Naivas')
df_quickmart = process_folder('../Data/Quickmart', 'Quickmart')

# Merge into one Master DataFrame
df_master = pd.concat([df_naivas, df_quickmart], ignore_index=True)

# Sort by date
df_master = df_master.sort_values(['Date', 'Store']).reset_index(drop=True)

print("\n" + "="*70)
print("Data Extraction Complete!")
print("="*70)
print(f"Total Records: {len(df_master):,}")
print(f"Stores: {df_master['Store'].unique()}")
print(f"Date Range: {df_master['Date'].min()} to {df_master['Date'].max()}")
print(f"DataFrame Shape: {df_master.shape}")

Starting Data Extraction...

Processing Naivas data from: Data/Naivas
✓ Processed: Naivas_CPI_2026-02-13.xlsx (886 rows)
✓ Processed: Naivas_CPI_2026-03-06.xlsx (915 rows)
✓ Processed: Naivas_CPI_2026-03-17.xlsx (4533 rows)
✓ Processed: Naivas_CPI_2026-02-26.xlsx (952 rows)
✓ Processed: Naivas_CPI_2026-03-11.xlsx (915 rows)
✓ Processed: Naivas_CPI_2026-03-04.xlsx (4903 rows)
✓ Processed: Naivas_CPI_2026-01-26.xlsx (4579 rows)
✓ Processed: Naivas_CPI_2026-02-24.xlsx (803 rows)
✓ Processed: Naivas_CPI_2026-02-17.xlsx (4644 rows)
✓ Processed: Naivas_CPI_2026-03-31.xlsx (753 rows)
✓ Processed: Naivas_CPI_2026-01-19.xlsx (3799 rows)
✓ Processed: Naivas_CPI_2026-02-16.xlsx (773 rows)
✓ Processed: Naivas_CPI_2026-01-27.xlsx (818 rows)
✓ Processed: Naivas_CPI_2026-02-03.xlsx (820 rows)
✓ Processed: Naivas_CPI_2026-03-30.xlsx (753 rows)
✓ Processed: Naivas_CPI_2026-02-12.xlsx (4101 rows)
✓ Processed: Naivas_CPI_2026-02-11.xlsx (777 rows)
✓ Processed: Naivas_CPI_2026-01-12.xlsx (4276 rows)
✓ Pro

## 4. Transform - Data Cleaning & Standardization

In [10]:
print("="*70)
print("Data Cleaning & Transformation")
print("="*70)

# Step 1: Combine overlapping columns (handling both uppercase and lowercase)
df_master['Final_Name'] = df_master['Name'].fillna(df_master['product_name'])
df_master['Final_Category'] = df_master['category'].fillna(df_master['Product_Category'])
df_master['Final_Unit'] = df_master['Unit'].fillna(df_master['unit'])
df_master['Final_Qty'] = df_master['Quantity'].fillna(df_master['quantity'])

print("✓ Step 1: Combined overlapping columns")

# Step 2: Clean Price column - handles both numeric and text "KES XX.XX" formats
def clean_price(row):
    """Clean and extract numeric price from both formats"""
    # Try Price column first (can be numeric or text)
    if pd.notna(row['Price']):
        price_val = row['Price']
        # If already numeric, return it
        if isinstance(price_val, (int, float)):
            return float(price_val)
        # If text, clean it
        price_str = str(price_val)
        # Remove "KES", "KSH", non-breaking spaces (\xa0), commas, regular spaces
        price_str = price_str.replace('KES', '').replace('KSH', '').replace('\xa0', '').replace(',', '').strip()
        try:
            return float(price_str)
        except:
            pass
    
    # Try lowercase price column (Quickmart format)
    if pd.notna(row['price']):
        price_str = str(row['price'])
        # Remove "KES", "KSH", non-breaking spaces, currency symbols, commas, and spaces
        price_str = price_str.replace('KES', '').replace('KSH', '').replace('\xa0', '').replace(',', '').strip()
        try:
            return float(price_str)
        except:
            pass
    
    return None

print("\n🧹 Step 2: Cleaning prices (removing 'KES' prefix, converting to numeric)...")
df_master['Final_Price'] = df_master.apply(clean_price, axis=1)
print("✓ Price cleaning complete")

# Step 3: Create clean DataFrame with standardized columns
df_clean = df_master[['Date', 'Store', 'Final_Name', 'Final_Price', 'Final_Category', 'Final_Unit', 'Final_Qty', 'Source_File']].copy()

# Rename for clarity
df_clean.columns = ['Date', 'Store', 'Product_Name', 'Price', 'Category', 'Unit', 'Quantity', 'Source_File']
print("✓ Step 3: Created clean DataFrame with standardized columns")

# Step 4: Data Quality Report
print("\n" + "="*70)
print("Transformation Complete!")
print("="*70)
print(f"Total Records: {len(df_clean):,}")
print(f"Records with valid prices: {df_clean['Price'].notna().sum():,}")
print(f"Records with missing prices: {df_clean['Price'].isna().sum():,}")

print("\n📊 Price Statistics by Store:")
print(df_clean.groupby('Store')['Price'].agg(['count', 'mean', 'min', 'max']))

Data Cleaning & Transformation
✓ Step 1: Combined overlapping columns

🧹 Step 2: Cleaning prices (removing 'KES' prefix, converting to numeric)...
✓ Price cleaning complete
✓ Step 3: Created clean DataFrame with standardized columns

Transformation Complete!
Total Records: 622,864
Records with valid prices: 622,864
Records with missing prices: 0

📊 Price Statistics by Store:
            count         mean  min        max
Store                                         
Naivas     193787  2673.144153  0.0   285995.0
Quickmart  429077  2524.026328  1.0  1099995.0


In [11]:
# Preview cleaned data
print("Sample of Cleaned Data:")
print("\n🏪 Naivas Sample:")
print(df_clean[df_clean['Store'] == 'Naivas'].head())

print("\n🏪 Quickmart Sample:")
print(df_clean[df_clean['Store'] == 'Quickmart'].head())

Sample of Cleaned Data:

🏪 Naivas Sample:
        Date   Store            Product_Name   Price Category Unit Quantity  \
0 2026-01-01  Naivas  Lea Premium Maize Meal   184.0      NaN   Kg      2.0   
1 2026-01-01  Naivas    Sunrice Pishori Rice  1459.0      NaN   Kg      5.0   
2 2026-01-01  Naivas     Daawat Deli Basmati   630.0      NaN   Kg      2.0   
3 2026-01-01  Naivas          Daawat Biryani  1199.0      NaN   Kg      5.0   
4 2026-01-01  Naivas  Ajab Home Baking Flour   172.0      NaN   Kg      2.0   

                  Source_File  
0  Naivas_CPI_2026-01-01.xlsx  
1  Naivas_CPI_2026-01-01.xlsx  
2  Naivas_CPI_2026-01-01.xlsx  
3  Naivas_CPI_2026-01-01.xlsx  
4  Naivas_CPI_2026-01-01.xlsx  

🏪 Quickmart Sample:
           Date      Store                           Product_Name  Price  \
9914 2026-01-05  Quickmart            Toni Glass Tonic Water Pear   70.0   
9915 2026-01-05  Quickmart  Toni Glass Tonic Water Virgin G&T250M   70.0   
9916 2026-01-05  Quickmart              Fa

## 5. Load - Insert into Data Warehouse

In [13]:
# Optional: Delete existing database for a fresh start
# Uncomment the lines below if you want to start fresh
import os
if os.path.exists('../data_warehouse.db'):
    os.remove('../data_warehouse.db')
#     print("✓ Existing database deleted")

**Note:** If you get an "already exists" error, you have two options:
1. **Append data**: Skip the schema initialization (comment out `etl.load_data()` and use custom loading)
2. **Fresh start**: Delete `data_warehouse.db` and re-run

In [14]:
# Import the ETL module
import sys, os
sys.path.insert(0, os.path.abspath('..'))
from scripts.etl_to_warehouse import DataWarehouseETL

# Initialize ETL pipeline
etl = DataWarehouseETL('../data_warehouse.db')

# Load data into warehouse
print("\n" + "="*70)
print("Loading data into Data Warehouse...")
print("="*70)

etl.load_data(df_clean)

print("\n" + "="*70)
print("🎉 ETL PIPELINE COMPLETE!")
print("="*70)
print("✓ Data extracted from files")
print("✓ Data cleaned and transformed")
print("✓ Data loaded into warehouse")
print("\nNext step: Run warehouse_queries.ipynb for analysis")


Loading data into Data Warehouse...

DATA WAREHOUSE ETL PIPELINE
✓ Connected to database: data_warehouse.db

Initializing Data Warehouse Schema
✓ Schema initialized successfully

Loading Dimension Tables:
----------------------------------------------------------------------
✓ Loaded 2 stores into dim_stores
✓ Loaded 8 categories into dim_categories
✓ Loaded 76 dates into dim_dates
✓ Loaded 20232 products into dim_products

Loading Fact Table:
----------------------------------------------------------------------

Loading fact_prices...
  Processed 10,000 / 622,864 records...
  Processed 20,000 / 622,864 records...
  Processed 30,000 / 622,864 records...
  Processed 40,000 / 622,864 records...
  Processed 50,000 / 622,864 records...
  Processed 60,000 / 622,864 records...
  Processed 70,000 / 622,864 records...
  Processed 80,000 / 622,864 records...
  Processed 90,000 / 622,864 records...
  Processed 100,000 / 622,864 records...
  Processed 110,000 / 622,864 records...
  Processed 12

## 6. Optional - Export Cleaned Data

In [ ]:
# Save cleaned data to CSV
output_path = '../Data/master_data_clean.csv'
df_clean.to_csv(output_path, index=False)
print(f"✓ Cleaned data exported to: {output_path}")